# SARIMAX — Fair 4-Fold Comparison

This notebook uses the same Aug/Nov/Feb/May 2025–2026 validation folds as the Prophet/XGBoost comparison, performs SARIMAX order selection only on data before the first fold, evaluates rolling next-24-hour forecasts, adds R², checkpoints fold results, and keeps June 2026 locked by default.

In [ ]:
# If pmdarima is missing in your environment, uncomment:
# %pip install -q pmdarima


In [ ]:

import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from pmdarima import auto_arima
from pmdarima.arima import ndiffs, nsdiffs
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

FINAL_TEST_START = pd.Timestamp("2026-06-01 00:00:00")

CV_FOLDS = [
    ("aug_2025", pd.Timestamp("2025-08-01 00:00:00"), pd.Timestamp("2025-08-31 23:00:00")),
    ("nov_2025", pd.Timestamp("2025-11-01 00:00:00"), pd.Timestamp("2025-11-30 23:00:00")),
    ("feb_2026", pd.Timestamp("2026-02-01 00:00:00"), pd.Timestamp("2026-02-28 23:00:00")),
    ("may_2026", pd.Timestamp("2026-05-01 00:00:00"), pd.Timestamp("2026-05-31 23:00:00")),
]

EXOG_COLS = [
    "apparent_temperature",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "is_holiday",
    "week_sin",
    "week_cos",
    "econ_industrial_production_index_lag1m",
    "econ_gdp_index_lag1m",
]

# Keep teammate's original practical design: SARIMAX is fit on the most recent year.
TRAIN_WINDOW_HOURS = 24 * 365

# Order search uses only data BEFORE the first validation fold.
ORDER_SEARCH_LOOKBACK_HOURS = 24 * 90

# Rolling next-24-hour evaluation.
FORECAST_HORIZON = 24

# Leave June locked until all models are compared.
RUN_FINAL_JUNE_TEST = False

OUTPUT_DIR = (
    Path("/kaggle/working/sarimax_outputs")
    if Path("/kaggle/working").exists()
    else Path("sarimax_outputs")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# DATA
# ============================================================

def find_input_csv():
    if Path("/kaggle/input").exists():
        matches = list(Path("/kaggle/input").rglob("master_training_data.csv"))
    else:
        matches = list(Path(".").rglob("master_training_data.csv"))

    if not matches:
        raise FileNotFoundError(
            "master_training_data.csv not found. "
            "On Kaggle, attach the existing master dataset using Add Input."
        )

    preferred = [p for p in matches if "master-train-data-uk-demand" in str(p)]
    return preferred[0] if preferred else matches[0]


def load_full_data():
    input_path = find_input_csv()
    print("Using:", input_path)

    df = pd.read_csv(
        input_path,
        parse_dates=["timestamp"],
        low_memory=False,
    )

    df = (
        df
        .dropna(subset=["timestamp", "demand_mw"])
        .sort_values("timestamp")
        .drop_duplicates(subset=["timestamp"], keep="last")
        .reset_index(drop=True)
    )

    # Deterministic weekly cyclical features.
    df["hour_of_week"] = df["day_of_week"] * 24 + df["hour"]
    df["week_sin"] = np.sin(2 * np.pi * df["hour_of_week"] / 168)
    df["week_cos"] = np.cos(2 * np.pi * df["hour_of_week"] / 168)

    # Keep only needed columns.
    keep = ["timestamp", "demand_mw"] + EXOG_COLS
    missing = [c for c in keep if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df[keep].copy()

    # Numeric conversion + forward-fill only (safe direction in time).
    for c in EXOG_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[EXOG_COLS] = df[EXOG_COLS].ffill()

    # Confirm hourly continuity.
    gaps = df["timestamp"].diff().dropna()
    if not gaps.eq(pd.Timedelta(hours=1)).all():
        raise ValueError("Dataset is not perfectly continuous hourly.")

    df = df.set_index("timestamp")
    df.index.freq = "h"

    print("Rows:", len(df))
    print("Range:", df.index.min(), "to", df.index.max())

    dev = df[df.index < FINAL_TEST_START].copy()
    june = df[df.index >= FINAL_TEST_START].copy()

    print("Development:", len(dev), dev.index.min(), "to", dev.index.max())
    print("FINAL TEST:", len(june), june.index.min(), "to", june.index.max())
    print("June 2026 remains LOCKED.")

    return dev, june


# ============================================================
# METRICS
# ============================================================

def get_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    mask = np.abs(actual) > 1e-8
    mape = np.mean(
        np.abs((actual[mask] - predicted[mask]) / actual[mask])
    ) * 100

    r2 = r2_score(actual, predicted)

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "mape": float(mape),
        "r2": float(r2),
    }


# ============================================================
# ORDER SEARCH — RUN ONCE, BEFORE FIRST CV FOLD
# ============================================================

def tune_order(dev):
    first_fold_start = CV_FOLDS[0][1]

    search_base = dev[dev.index < first_fold_start].copy()
    search_data = search_base.tail(ORDER_SEARCH_LOOKBACK_HOURS).copy()

    if len(search_data) < ORDER_SEARCH_LOOKBACK_HOURS:
        print("Warning: order-search window shorter than requested.")

    y_search = np.log(search_data["demand_mw"])

    # Stationarity diagnostics on data available before first fold only.
    adf_p = adfuller(y_search, autolag="AIC", maxlag=48)[1]
    d = ndiffs(y_search, test="kpss", max_d=2)
    D = nsdiffs(y_search, m=24, test="ocsb", max_D=1)

    print(f"ADF p-value: {adf_p:.4f}, d={d}, D={D}")

    X_search = search_data[EXOG_COLS].copy()

    # Drop exogenous columns with no variance in this search window.
    ok_cols = X_search.columns[X_search.std() > 1e-8].tolist()

    X_search = X_search[ok_cols]
    X_mean = X_search.mean()
    X_std = X_search.std().replace(0, 1)
    X_search = (X_search - X_mean) / X_std

    print("Order search exogenous columns:", ok_cols)

    t0 = time.time()

    model = auto_arima(
        y_search,
        X=X_search,
        start_p=0,
        start_q=0,
        max_p=2,
        max_q=2,
        d=d,
        start_P=0,
        start_Q=0,
        max_P=1,
        max_Q=1,
        D=D,
        m=24,
        seasonal=True,
        stepwise=True,
        error_action="ignore",
        suppress_warnings=True,
        information_criterion="aic",
        maxiter=50,
    )

    elapsed = time.time() - t0

    order_info = {
        "order": list(model.order),
        "seasonal_order": list(model.seasonal_order),
        "aic": float(model.aic()),
        "exog_cols": ok_cols,
        "log_transform": True,
        "order_search_end": str(first_fold_start - pd.Timedelta(hours=1)),
        "order_search_seconds": elapsed,
    }

    with open(OUTPUT_DIR / "sarimax_order.json", "w") as f:
        json.dump(order_info, f, indent=2)

    print("\nSelected order:", model.order)
    print("Selected seasonal order:", model.seasonal_order)
    print(f"Order search completed in {elapsed:.1f}s")

    return order_info


# ============================================================
# ONE FOLD
# ============================================================

def evaluate_fold(dev, order_info, fold_name, valid_start, valid_end):
    order = tuple(order_info["order"])
    seasonal_order = tuple(order_info["seasonal_order"])
    exog_cols = list(order_info["exog_cols"])

    train_full = dev[dev.index < valid_start].copy()
    valid = dev[(dev.index >= valid_start) & (dev.index <= valid_end)].copy()

    # Match teammate's original design: latest year only for SARIMAX fit.
    train = train_full.tail(TRAIN_WINDOW_HOURS).copy()

    # Drop rows that still have missing values.
    train = train.dropna(subset=["demand_mw"] + exog_cols)
    valid = valid.dropna(subset=["demand_mw"] + exog_cols)

    X_train = train[exog_cols].copy()
    X_mean = X_train.mean()
    X_std = X_train.std().replace(0, 1)

    X_train = (X_train - X_mean) / X_std
    X_valid = (valid[exog_cols] - X_mean) / X_std

    y_train_log = np.log(train["demand_mw"])

    print(
        f"\n{fold_name}: train={len(train):,}, valid={len(valid):,}, "
        f"{valid_start} -> {valid_end}"
    )

    model = SARIMAX(
        y_train_log,
        exog=X_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    t0 = time.time()
    result = model.fit(disp=False, maxiter=100)
    fit_seconds = time.time() - t0

    print(f"Fit completed in {fit_seconds:.1f}s")

    # Rolling next-24-hour evaluation.
    preds_log = []
    pred_index = []
    current_result = result

    n_blocks = len(valid) // FORECAST_HORIZON

    for i in range(n_blocks):
        block = valid.iloc[
            i * FORECAST_HORIZON:(i + 1) * FORECAST_HORIZON
        ]
        X_block = X_valid.iloc[
            i * FORECAST_HORIZON:(i + 1) * FORECAST_HORIZON
        ]

        forecast = current_result.get_forecast(
            steps=FORECAST_HORIZON,
            exog=X_block,
        )

        preds_log.extend(forecast.predicted_mean.values)
        pred_index.extend(block.index)

        # Update model state with newly observed actual demand; no parameter refit.
        current_result = current_result.append(
            np.log(block["demand_mw"]),
            exog=X_block,
            refit=False,
        )

    preds = pd.Series(
        np.exp(np.asarray(preds_log)),
        index=pred_index,
        name="prediction",
    )

    actual = valid.loc[preds.index, "demand_mw"]

    metrics = get_metrics(actual.values, preds.values)

    print({
        **{k: round(v, 4) for k, v in metrics.items()},
        "fit_seconds": round(fit_seconds, 1),
    })

    pred_df = pd.DataFrame({
        "timestamp": preds.index,
        "actual_demand_mw": actual.values,
        "predicted_demand_mw": preds.values,
        "fold": fold_name,
    })

    return {
        "fold": fold_name,
        "train_rows": len(train),
        "valid_rows": len(preds),
        "fit_seconds": fit_seconds,
        **metrics,
    }, pred_df


# ============================================================
# MAIN 4-FOLD CV
# ============================================================

def run_cv(dev):
    order_info = tune_order(dev)

    fold_rows = []
    all_predictions = []

    for fold_name, valid_start, valid_end in CV_FOLDS:
        row, pred_df = evaluate_fold(
            dev,
            order_info,
            fold_name,
            valid_start,
            valid_end,
        )

        fold_rows.append(row)
        all_predictions.append(pred_df)

        pd.DataFrame(fold_rows).to_csv(
            OUTPUT_DIR / "sarimax_cv_folds_checkpoint.csv",
            index=False,
        )

    folds_df = pd.DataFrame(fold_rows)

    summary = {
        "model": "SARIMAX",
        "order": order_info["order"],
        "seasonal_order": order_info["seasonal_order"],
        "exog_cols": order_info["exog_cols"],
        "forecast_horizon_hours": FORECAST_HORIZON,
        "train_window_hours": TRAIN_WINDOW_HOURS,
        "mean_mae": float(folds_df["mae"].mean()),
        "mean_rmse": float(folds_df["rmse"].mean()),
        "mean_mape": float(folds_df["mape"].mean()),
        "mean_r2": float(folds_df["r2"].mean()),
        "std_rmse": float(folds_df["rmse"].std()),
        "worst_fold_rmse": float(folds_df["rmse"].max()),
        "min_fold_r2": float(folds_df["r2"].min()),
        "final_test_start": str(FINAL_TEST_START),
    }

    folds_df.to_csv(
        OUTPUT_DIR / "sarimax_cv_folds.csv",
        index=False,
    )

    pd.concat(all_predictions, ignore_index=True).to_csv(
        OUTPUT_DIR / "sarimax_cv_predictions.csv",
        index=False,
    )

    with open(OUTPUT_DIR / "sarimax_cv_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\n=== SARIMAX 4-FOLD CV SUMMARY ===")
    print(folds_df)
    print("\nSummary:")
    print(json.dumps(summary, indent=2))

    return order_info, folds_df, summary


# ============================================================
# OPTIONAL FINAL JUNE TEST
# ============================================================

def final_june_eval(dev, june, order_info):
    if not RUN_FINAL_JUNE_TEST:
        print("\nRUN_FINAL_JUNE_TEST = False — June remains untouched.")
        return None

    order = tuple(order_info["order"])
    seasonal_order = tuple(order_info["seasonal_order"])
    exog_cols = list(order_info["exog_cols"])

    train = dev.tail(TRAIN_WINDOW_HOURS).copy()
    test = june.copy()

    train = train.dropna(subset=["demand_mw"] + exog_cols)
    test = test.dropna(subset=["demand_mw"] + exog_cols)

    X_train = train[exog_cols]
    X_mean = X_train.mean()
    X_std = X_train.std().replace(0, 1)

    X_train = (X_train - X_mean) / X_std
    X_test = (test[exog_cols] - X_mean) / X_std

    model = SARIMAX(
        np.log(train["demand_mw"]),
        exog=X_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    result = model.fit(disp=False, maxiter=100)

    preds_log = []
    pred_index = []
    current_result = result

    n_blocks = len(test) // FORECAST_HORIZON

    for i in range(n_blocks):
        block = test.iloc[
            i * FORECAST_HORIZON:(i + 1) * FORECAST_HORIZON
        ]
        X_block = X_test.iloc[
            i * FORECAST_HORIZON:(i + 1) * FORECAST_HORIZON
        ]

        forecast = current_result.get_forecast(
            steps=FORECAST_HORIZON,
            exog=X_block,
        )

        preds_log.extend(forecast.predicted_mean.values)
        pred_index.extend(block.index)

        current_result = current_result.append(
            np.log(block["demand_mw"]),
            exog=X_block,
            refit=False,
        )

    preds = pd.Series(
        np.exp(np.asarray(preds_log)),
        index=pred_index,
    )

    actual = test.loc[preds.index, "demand_mw"]

    metrics = get_metrics(actual.values, preds.values)

    with open(OUTPUT_DIR / "sarimax_final_june_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    pd.DataFrame({
        "timestamp": preds.index,
        "actual_demand_mw": actual.values,
        "predicted_demand_mw": preds.values,
    }).to_csv(
        OUTPUT_DIR / "sarimax_final_june_predictions.csv",
        index=False,
    )

    print("\n=== FINAL JUNE SARIMAX RESULT ===")
    print(metrics)

    return metrics




In [ ]:
dev, june = load_full_data()
order_info, folds_df, summary = run_cv(dev)
final_june_eval(dev, june, order_info)
